# Transformer framework

## 1.0. Environment

In [ ]:
pip install 'accelerate>=0.26.0'

In [1]:
import warnings
warnings.filterwarnings('ignore')

### if only cpu or not cuda support 

In [3]:
import os
os.environ["TOEKNIZERS_PARALLELISM"] = "false"

### import transformer

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

### import llama

In [65]:
from llama_cpp import Llama

### offline data
- if cannot visit the hugging face directly
- Phi-3-mini-4k-instruct/files:
  - config.json
  - generation_config.json
  - model-00001-of-00002.safetensors: 4.97GB
  - model-00002-of-00002.safetensors: 2.67GB
  - model.safetensors.index.json
  - tokenizer_config.json
  - tokenizer.json
  - tokenizer.model
- Phi-3-mini-4k-instruct-q4/files:
  - Phi-3-mini-4k-instruct-q4.gguf: 2.39GB   

In [7]:
model_path = "./data/microsoft/Phi-3-mini-4k-instruct"
model_path_q4 = "./data/microsoft/Phi-3-mini-4k-instruct-q4/Phi-3-mini-4k-instruct-q4.gguf"

## 2.0. Transformer

### 2.0.1. Load tokenizer

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_path)

#### test the tokenizer

In [11]:
sentence_test = "hello world!"

In [13]:
token_ids = tokenizer(sentence_test).input_ids

In [15]:
print(token_ids)

[22172, 3186, 29991]


In [17]:
for token_id in token_ids:
    print(tokenizer.decode(token_id))

hello
world
!


### 2.0.2. Load model

#### compress data for cuda
- 4bit: 1/4 memory
- 8bit: 1/2 memory

In [ ]:
# quantization_config = BitsAndBytesConfig(load_in_4bit = True)

#### modeling

In [19]:
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    #quantization_config = quantization_config,
    device_map = "cpu", # "cuda", "auto"
    torch_dtype = "auto",
    trust_remote_code = False, # True for cuda and flash-attention
    attn_implementation = "eager", # not support flash-attention
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [21]:
prompt = "The capital of France is"

### 2.0.3.0. No pipeline

In [23]:
inputs = tokenizer(prompt, return_tensors="pt")

In [25]:
outputs = model.generate(**inputs)

You are not running the flash-attention implementation, expect numerical differences.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


In [27]:
result = tokenizer.decode(outputs[0])

In [29]:
print(result)

The capital of France is Paris.


### Response:The capital of France is Paris


### 2.0.3.1. Create a pipeline

In [31]:
generator = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    return_full_text = False,
    max_new_tokens = 50,
    do_sample = False,
)

In [33]:
#prompt = "write an email apologizing to sarah for the tragic gardending mishap. Explain how it happened"
output = generator(prompt)
print(output[0]['generated_text'])

 Paris.


### Response:The capital of France is Paris.


### 2.0.4. Show the details of the model

In [35]:
model

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
          (rotary_emb): Phi3RotaryEmbedding()
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLU()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
  )
 

In [37]:
model.model

Phi3Model(
  (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
  (embed_dropout): Dropout(p=0.0, inplace=False)
  (layers): ModuleList(
    (0-31): 32 x Phi3DecoderLayer(
      (self_attn): Phi3Attention(
        (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        (rotary_emb): Phi3RotaryEmbedding()
      )
      (mlp): Phi3MLP(
        (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
        (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
        (activation_fn): SiLU()
      )
      (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
      (resid_attn_dropout): Dropout(p=0.0, inplace=False)
      (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
    )
  )
  (norm): Phi3RMSNorm((3072,), eps=1e-05)
)

In [39]:
model.model.layers[0]

Phi3DecoderLayer(
  (self_attn): Phi3Attention(
    (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
    (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (mlp): Phi3MLP(
    (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
    (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
    (activation_fn): SiLU()
  )
  (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
  (resid_attn_dropout): Dropout(p=0.0, inplace=False)
  (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
  (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
)

### 2.0.5. Decompose the generation 

In [41]:
prompt = "The capital of France is"

In [43]:
# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors = "pt").input_ids
input_ids

tensor([[ 450, 7483,  310, 3444,  338]])

In [47]:
model_output = model.model(input_ids)

In [49]:
model_output[0].shape

torch.Size([1, 5, 3072])

In [51]:
lm_head_output = model.lm_head(model_output[0])

In [53]:
lm_head_output.shape

torch.Size([1, 5, 32064])

In [59]:
token_id = lm_head_output[0, -1].argmax(-1)
token_id

tensor(3681)

In [61]:
tokenizer.decode(token_id)

'Paris'

## 2.1. Llama

In [67]:
llm = Llama(
    model_path = model_path_q4, 
    n_ctx = 4096, # the length of the context
    n_gpu_layers = 0, # = 0 cpu
    n_thread = 8,
)


ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 0.168 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   MTL0 (Intel(R) Iris(TM) Plus Graphics 645)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal3  (5001)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = false
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has tensor            = false
ggml_metal_device_init: use residency sets    = true
ggml_metal_device_init: use shared buffers    = true
ggml_metal_device_init: recommendedMaxWorkingSetSize  =  1610.61 MB
llama_model_loader: loaded meta data with 24 key-value pairs and 195 tensors from ./data/microsoft/P

In [69]:
prompt = "The capital of France is"

In [71]:
output = llm(
    prompt,
    max_tokens = 50,
    echo = True,
)

llama_perf_context_print:        load time =    5133.84 ms
llama_perf_context_print: prompt eval time =    5131.89 ms /     6 tokens (  855.31 ms per token,     1.17 tokens per second)
llama_perf_context_print:        eval time =    5657.96 ms /    49 runs   (  115.47 ms per token,     8.66 tokens per second)
llama_perf_context_print:       total time =   10805.81 ms /    55 tokens
llama_perf_context_print:    graphs reused =         48


In [73]:
print(output['choices'][0]['text'])

The capital of France is Paris. It is known for its rich history, art, and culture. Paris is home to famous landmarks such as the Eiffel Tower, Notre-Dame Cathedral, and the Louvre Museum, which houses the Mona Lisa.


## 2.3. Transformers vs Llama
- Core: pytorch -> c++
- Architexture:
  - LayerNorm:  after self-attention -> before self-attention(Pre-Norm)
  - Activation Function: ReLU -> SwiGLU
  - Positional Embedding: -> RoPE